In [3]:
# %pip install pinecone

In [4]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

C:\Users\91904\AppData\Roaming\Python\Python312\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [5]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PDF_PATH = PROJECT_DIR / "data" / "Attention Is All You Need.pdf"
EMBEDDING_MODEL = "intfloat/e5-base-v2"
EMBEDDING_DIMENSION = 768
INDEX_NAME = "attention-is-all-you-need"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
PINECONE_METRIC = "cosine"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
UPSERT_BATCH_SIZE = 100

print(f"Project directory: {PROJECT_DIR}")
print(f"PDF path: {PDF_PATH}")

Project directory: c:\Users\91904\Desktop\ksfe\AI\CODE\rag-fastapi-chatbot
PDF path: c:\Users\91904\Desktop\ksfe\AI\CODE\rag-fastapi-chatbot\data\Attention Is All You Need.pdf


In [6]:
load_dotenv(PROJECT_DIR / ".env")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise RuntimeError("PINECONE_API_KEY is missing from the project .env file.")
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF file was not found: {PDF_PATH}")

print("Environment and PDF path are ready.")

Environment and PDF path are ready.


In [7]:
pc = Pinecone(api_key=PINECONE_API_KEY)
print("Connected to the Pinecone control plane.")

Connected to the Pinecone control plane.


In [8]:
reader = PdfReader(PDF_PATH)
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()
    if page_text and page_text.strip():
        pages.append((page_number, page_text.strip()))

if not pages:
    raise ValueError(f"No text could be extracted from {PDF_PATH}")

print(f"Extracted text from {len(pages)} of {len(reader.pages)} PDF pages.")

Extracted text from 15 of 15 PDF pages.


In [9]:
chunks = []
page_numbers = []

for page_number, page_text in pages:
    words = page_text.split()
    start = 0
    while start < len(words):
        end = start + CHUNK_SIZE
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
            page_numbers.append(page_number)
        start += CHUNK_SIZE - CHUNK_OVERLAP

if not chunks:
    raise ValueError("No document chunks were created from the PDF.")

print(f"Created {len(chunks)} chunks.")

Created 24 chunks.


In [10]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded document embedding model: {EMBEDDING_MODEL}")

Loaded document embedding model: intfloat/e5-base-v2


In [11]:
passages = [f"passage: {chunk}" for chunk in chunks]
embeddings = embedding_model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True,
)

if len(embeddings) != len(chunks):
    raise RuntimeError("The number of generated embeddings does not match the number of chunks.")

print(f"Generated {len(embeddings)} document embeddings.")

Batches: 100%|██████████| 1/1 [00:29<00:00, 29.53s/it]

Generated 24 document embeddings.


In [12]:
existing_indexes = pc.list_indexes().names()
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric=PINECONE_METRIC,
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print(f"Creating Pinecone index '{INDEX_NAME}'...")
    ready = False
    for _ in range(120):
        description = pc.describe_index(INDEX_NAME)
        if description.status["ready"]:
            ready = True
            break
        time.sleep(1)
    if not ready:
        raise TimeoutError(f"Pinecone index '{INDEX_NAME}' did not become ready within 120 seconds.")
else:
    print(f"Pinecone index '{INDEX_NAME}' already exists.")

description = pc.describe_index(INDEX_NAME)
if description.dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Index dimension is {description.dimension}; expected {EMBEDDING_DIMENSION}."
    )
if description.metric != PINECONE_METRIC:
    raise ValueError(f"Index metric is {description.metric!r}; expected {PINECONE_METRIC!r}.")

print(f"Index '{INDEX_NAME}' is ready with dimension {description.dimension} and metric {description.metric}.")

Pinecone index 'attention-is-all-you-need' already exists.
Index 'attention-is-all-you-need' is ready with dimension 768 and metric cosine.


In [13]:
index = pc.Index(INDEX_NAME)
print(f"Connected to Pinecone index '{INDEX_NAME}'.")

Connected to Pinecone index 'attention-is-all-you-need'.


In [14]:
vectors = [
    {
        "id": f"chunk_{chunk_number}",
        "values": embedding.tolist(),
        "metadata": {"text": chunk, "page": page_number},
    }
    for chunk_number, (chunk, page_number, embedding) in enumerate(
        zip(chunks, page_numbers, embeddings)
    )
]

if any(len(vector["values"]) != EMBEDDING_DIMENSION for vector in vectors):
    raise ValueError("At least one embedding has the wrong dimension for the Pinecone index.")

print(f"Prepared {len(vectors)} Pinecone vectors.")

Prepared 24 Pinecone vectors.


In [15]:
# This dedicated paper index is fully refreshed when this upload cell is run.
# Clearing it first prevents stale chunks when the replacement PDF has fewer chunks.
current_stats = index.describe_index_stats()
current_count = (
    current_stats.get("total_vector_count", 0)
    if isinstance(current_stats, dict)
    else getattr(current_stats, "total_vector_count", 0)
)
if current_count:
    index.delete(delete_all=True)
    for _ in range(120):
        current_stats = index.describe_index_stats()
        current_count = (
            current_stats.get("total_vector_count", 0)
            if isinstance(current_stats, dict)
            else getattr(current_stats, "total_vector_count", 0)
        )
        if current_count == 0:
            break
        time.sleep(1)
    if current_count != 0:
        raise TimeoutError("Existing Pinecone vectors were not cleared within 120 seconds.")

for start in range(0, len(vectors), UPSERT_BATCH_SIZE):
    batch = vectors[start : start + UPSERT_BATCH_SIZE]
    index.upsert(vectors=batch)
    print(f"Uploaded vectors {start + 1}-{start + len(batch)} of {len(vectors)}.")

Uploaded vectors 1-24 of 24.


In [16]:
expected_count = len(vectors)
verified_count = 0
for _ in range(120):
    stats = index.describe_index_stats()
    verified_count = (
        stats.get("total_vector_count", 0)
        if isinstance(stats, dict)
        else getattr(stats, "total_vector_count", 0)
    )
    if verified_count == expected_count:
        break
    time.sleep(1)

print(f"Expected vectors: {expected_count}")
print(f"Pinecone vectors: {verified_count}")
if verified_count != expected_count:
    raise RuntimeError(
        f"Pinecone verification failed: expected {expected_count} vectors, found {verified_count}."
    )
print("Pinecone indexing completed and verified.")

Expected vectors: 24
Pinecone vectors: 24
Pinecone indexing completed and verified.
